# 06 — Сравнение теорий: `classic` / `karman` / `ktn_linear` / `ktn_full`

Лестница моделей одним ключом `[model] theory` (честные термины):

* **`classic`** — линейная теория Кирхгофа;
* **`karman`** — геометрически-**НЕЛИНЕЙНОЕ** решение Фёппля–Кармана (мембранное
  поле `N` ужестчает пластину);
* **`ktn_linear`** — **ЛИНЕЙНЫЕ** поправки сдвига/обжатия постобработкой (прежнее
  `ktn`, число-в-число);
* **`ktn_full`** — **ПОЛНАЯ нелинейная КТН**: Карман + `(I − h_ψ²Δ)L(Φ,w)`
  + нагрузочный член `−h_*²Δq_n` (`ktn_full.py`).

Каноническая пластина — защемлённый круг, неподвижная кромка. Безразмерно:
`E = h = a = 1` в разделе Кармана ⇒ `P̄ = q0`. Case-файлы:
`cases/ci/ktn_full_circle_clamped_uniform.toml`,
`cases/ci/ktn_full_circle_clamped_gaussian.toml`.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from plate_solver import benchmarks as bm
from plate_solver.config import Config
from plate_solver.geometry import make_circle
from plate_solver.ktn_full import KTNPlate
from plate_solver.membrane import KarmanPlate

nu = 0.3
dom = make_circle(1.0)


def solve_karman(P_bar, cls=KarmanPlate, h=1.0, Q=120):
    cfg = Config(E=1.0, h=h, nu=nu, a=1.0, q0=P_bar * h**4, p=12, Q=Q,
                 n_load_steps=max(2, int(round(P_bar / 2))),
                 karman_tol=1e-6, karman_max_iter=300)
    return cls.from_config(dom, cfg, bc_type="clamped", inplane_bc="immovable").solve_uniform()

## 1. Нагрузка–прогиб: три нелинейные теории и эталоны

`classic` — прямая; `karman` — загиб вниз (мембранное ужесточение), ложится на
ряды **Way**. Здесь `h/L = 1` (ТОЛСТАЯ пластина): при такой толщине поправка КТН
`O(h²/L²)` велика, и `ktn_full` заметно мягче `karman`. Для ТОНКИХ пластин
`ktn_full → karman` (п. 4), а под равномерной нагрузкой поправка КТН для
срединного прогиба живёт в ЛИЦЕВЫХ величинах (п. 2). При большой нагрузке —
асимптота мембраны **Hencky**.

In [ ]:
p_bars = np.array([0.5, 1.0, 2.0, 3.196, 4.561, 6.321, 8.635, 11.71])
w_karman, w_classic, w_ktn = [], [], []
for pb in p_bars:
    rk = solve_karman(pb, KarmanPlate)
    rt = solve_karman(pb, KTNPlate)
    w_karman.append(rk.w_max)
    w_classic.append(rk.w_max_classic)
    w_ktn.append(rt.w_max)
w_karman, w_classic, w_ktn = map(np.array, (w_karman, w_classic, w_ktn))
dev = float(np.max(np.abs(w_ktn - w_karman) / w_karman))
print(f"h/L=1 (толстая): ktn_full мягче karman на {dev*100:.1f}% — КТН существенна; "
      "для тонких пластин ktn_full → karman (п. 4)")

In [ ]:
way = bm.way_clamped_circle()
pb_fine = np.linspace(0.05, p_bars.max(), 200)
hencky = np.array([bm.hencky_center_deflection(x, nu) for x in pb_fine])

fig, ax = plt.subplots(figsize=(7.5, 5.2))
ax.plot(p_bars, w_classic, "o--", color="tab:blue", label="classic (Кирхгоф)")
ax.plot(p_bars, w_karman, "s-", color="tab:red", label="karman")
ax.plot(p_bars, w_ktn, "x:", color="tab:green", ms=9, label="ktn_full (h/L=1, толстая)")
ax.plot(way[:, 0], way[:, 1], "k*", ms=13, label="эталон Way")
ax.plot(pb_fine, hencky, ":", color="crimson", label="асимптота Hencky")
ax.set_xlabel("P̄ = p·a⁴/(E·h⁴)")
ax.set_ylabel("w / h")
ax.set_title("Нагрузка–прогиб: classic / karman / ktn_full и эталоны")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.show()

## 2. Подпись КТН под РАВНОМЕРНОЙ нагрузкой — в лицевых величинах

Под равномерной нагрузкой срединный прогиб `ktn_full` ≈ `karman` (`Δq = 0`), а
поправка сдвига/обжатия проявляется в ЛИЦЕВЫХ величинах: смещение контактирующей
лицевой `dh = u_c − w`. Толстая пластина (`h/L = 0.2`) — поправки `(h/L)²`
заметны. Интроспекция `h_ψ², h_*², h_c²` — `Result.thickness_params`.

In [ ]:
from plate_solver.dispatch import solve
from plate_solver.problem import Problem

res = solve(Problem.from_dict({
    "geometry": {"kind": "circle", "a": 1.0}, "bc": {"type": "clamped"},
    "load": {"type": "uniform", "q0": 8.0e-3},
    "model": {"theory": "ktn_full", "E": 1.0, "nu": 0.3, "h": 0.2, "n_load_steps": 4},
    "discretization": {"p": 12, "Q": 140, "grid_n": 60},
}))
tp = res.thickness_params()
print(f"h/L = {tp['h_over_L']:.3f}: h_ψ² = {tp['h_psi_sq']:.4e}, "
      f"h_*² = {tp['h_star_sq']:.4e}, h_c² = {tp['h_c_sq']:.4e}")
_, _, dh = res.faces_on_grid()
print(f"лицевое смещение max|dh| = {np.nanmax(np.abs(dh)):.4e}")

fig, ax = plt.subplots(figsize=(5.6, 5))
cs = ax.contourf(res.Xg, res.Yg, dh, levels=20, cmap="coolwarm")
fig.colorbar(cs, label="dh = u_c − w (лицевая подпись КТН)")
ax.set_aspect("equal")
ax.set_title("Подпись КТН под равномерной нагрузкой (лицевое смещение)")
plt.show()

## 3. Подпись КТН под НЕРАВНОМЕРНОЙ (гауссовой) нагрузкой — в срединном прогибе

При `Δq ≠ 0` проявляется член КТН `−h_*²Δq`: уже и СРЕДИННЫЙ прогиб `ktn_full`
отличается от классики. Эффект РАСТЁТ с сужением гауссовой нагрузки (`Δq ∝ 1/σ²`).
Малая нагрузка — линейный режим, виден чистый член (A).

In [ ]:
def mid_dev(sigma):
    base = {"geometry": {"kind": "circle", "a": 1.0}, "bc": {"type": "clamped"},
            "load": {"type": "gaussian", "q0": 2.0e-5, "x0": 0.0, "y0": 0.0, "sigma": sigma},
            "discretization": {"p": 12, "Q": 160, "grid_n": 40}}
    kt = solve(Problem.from_dict({**base, "model": {"theory": "ktn_full", "E": 1.0,
               "nu": 0.3, "h": 0.2, "n_load_steps": 1}}))
    cl = solve(Problem.from_dict({**base, "model": {"theory": "classic", "E": 1.0,
               "nu": 0.3, "h": 0.2}}))
    return abs(kt.w_max - cl.w_max) / cl.w_max

sigmas = np.array([0.45, 0.35, 0.25, 0.18, 0.13])
devs = np.array([mid_dev(s) for s in sigmas]) * 100
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sigmas, devs, "o-", color="tab:green")
ax.set_xlabel("σ (ширина гауссовой нагрузки)")
ax.set_ylabel("|w_ktn − w_classic| / w_classic, %")
ax.set_title("Член КТН (A) −h_*²Δq: срединная подпись растёт с локализацией")
ax.grid(alpha=0.3)
ax.invert_xaxis()
plt.show()
print("под равномерной (σ→∞, Δq=0) срединный эффект → 0")

## 4. Зависимость эффекта КТН от толщины (Gate R4)

При ФИКСИРОВАННОЙ безразмерной нагрузке `P̄` эффект `ktn_full / karman` для
срединного прогиба масштабируется как `(h/L)²` и гаснет при утоньшении —
редукция КТН → Карман (теорема T6).

In [ ]:
P_bar = 5.0
hs = np.array([0.25, 0.2, 0.15, 0.1, 0.05])
eff = []
for h in hs:
    rk = solve_karman(P_bar, KarmanPlate, h=h, Q=140)
    rt = solve_karman(P_bar, KTNPlate, h=h, Q=140)
    eff.append(abs(1.0 - rt.w_max / rk.w_max))
eff = np.array(eff)
fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(hs, eff, "o-", color="tab:purple", label="эффект КТН/Карман")
ax.loglog(hs, eff[0] * (hs / hs[0]) ** 2, "k--", alpha=0.6, label="наклон ~ h²")
ax.set_xlabel("h / L")
ax.set_ylabel("|1 − w_ktn/w_karman|")
ax.set_title("Гашение поправки КТН: O(h²/L²) (Gate R4)")
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.show()

## 5. Итог

| Теория | Доп. член | Где виден | Природа |
|---|---|---|---|
| `classic` | — | линейная жёсткость | линейный Кирхгоф |
| `karman` | мембранная связь `N:∇∇w` | загиб «нагрузка–прогиб» вниз | геом. нелинейность |
| `ktn_linear` | сдвиг+обжатие (линейные) | лицевые величины | поправки `(h/L)²` |
| `ktn_full` | `(I−h_ψ²Δ)L − h_*²Δq` | лицевые (равномерная); срединный (неравномерная) | полная нелинейная КТН |

**Оговорки.** Неподвижная кромка обязательна для эффекта Кармана; под равномерной
нагрузкой (`Δq = 0`) срединный `w` КТН ≈ Карман, подпись — в лицевых; под
неравномерной проявляется член `−h_*²Δq`; все КТН-поправки `O(h²/L²)` гаснут при
`h/L → 0` (редукция КТН→Карман→Кирхгоф, теорема T6). Разные ν эталонов
(приложение B). Полный нелинейный контакт (МОР поверх КТН) — направление
развития v0.6.0.